# multi_strat_048

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (72).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 6 |
| Detected functions | calculate_pvs_vms, calculate_samx, process_ticker, backtest_trades |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2015-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT'] # Add tickers you want

# Apply filtering
data = data[
(data["Date"] > = start_date) &
(data["Date"] < = end_date) &
(data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
def calculate_pvs_vms(data, atr_period = 14, volume_period = 14):
"""
Calculates Price Volatility Strength (PVS) and Volume Momentum Strength (VMS) separately.

Parameters:
- data: DataFrame with columns ['Date', 'Ticker', 'close', 'volume']
- atr_period: Lookback period for ATR calculation (default = 14)
- volume_period: Lookback period for average volume calculation (default = 14)

Returns:
- UpDated DataFrame with PVS and VMS indicators.
"""

# Sort data properly
data = data.sort_values(by = ['Ticker', 'Date']).copy()

# Compute Previous Close
data['Prev_Close'] = data.groupby('Ticker')['close'].shift(1)

# Compute True Range (TR) using Close Price Difference (simplified ATR approach)
data['TR'] = (data['close'] - data['Prev_Close']).abs()

# Compute ATR using rolling mean
data['ATR'] = data.groupby('Ticker')['TR'].transform(lambda x: x.rolling(atr_period).mean())

# Compute Average Volume over a rolling window
data['Avg_Volume'] = data.groupby('Ticker')['volume'].transform(lambda x: x.rolling(volume_period).mean())

# Compute Price Volatility Strength (PVS)
data['PVS'] = (data['close'] - data['Prev_Close']) / data['ATR']

# Compute Volume Momentum Strength (VMS)
data['VMS'] = (data['volume'] - data['Avg_Volume']) / data['Avg_Volume']

return data[['Date', 'Ticker', 'open', 'high', 'low', 'close', 'volume', 'PVS', 'VMS']]

# Example usage (Assuming you have a DataFrame named 'data' with required columns)
data = calculate_pvs_vms(data)

# Display result
data

In [ ]:
def calculate_samx(data, atr_period = 14, volume_period = 14):
"""
Calculates the Simplified AMX (SAMX) indicator based on price momentum and volume strength.

Parameters:
- data: DataFrame with columns ['Date', 'Ticker', 'open', 'high', 'low', 'close', 'volume']
- atr_period: Lookback period for ATR calculation (default = 14)
- volume_period: Lookback period for average volume calculation (default = 14)

Returns:
- Updated DataFrame with SAMX indicator
"""

# Ensure proper sorting
data = data.sort_values(by = ['Ticker', 'Date']).copy()

# Compute Previous Close using transform to maintain alignment
data['prev_close'] = data.groupby('Ticker')['close'].transform(lambda x: x.shift(1))

# Calculate True Range (TR)
data['TR'] = np.maximum(data['high'] - data['low'], 
np.maximum(abs(data['high'] - data['prev_close']), 
abs(data['low'] - data['prev_close'])))

# Compute ATR
data['ATR'] = data.groupby('Ticker')['TR'].transform(lambda x: x.rolling(atr_period, min_periods = 1).mean())

# Replace zero ATR values with a small constant to prevent division errors
data['ATR'].replace(0, np.finfo(float).eps, inplace = True)

# Compute Price Momentum (PMS)
data['PMS'] = data.groupby('Ticker')['close'].diff() / data['ATR']

# Compute SAMX
data['SAMX'] = data['PMS'] * data['VMS']

return data[['Date', 'Ticker', 'open', 'high', 'low', 'close', 'volume', 'prev_close', 'TR', 'ATR', 'PMS', 'PVS', 'VMS', 'SAMX']]

# Example usage (assuming `data` exists)
data = calculate_samx(data)
data

In [ ]:
from tqdm import tqdm
from joblib import Parallel, delayed

data["signal"] = 0 # Default value

# Long Entry Condition
data.loc[(data["SAMX"] < = 50) & (data["PVS"] > = 3) & (data["VMS"] > = 1), "signal"] = 1

# Short Entry Condition (More Strict)
data.loc[(data["SAMX"] > = -50) & (data["PVS"] < = -3) & (data["VMS"] > = 1), "signal"] = 2

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
"""Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
open_positions = {}
trades = []
cumulative_pnl = 0

for row in ticker_data:
timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["signal"]

# Check for open positions and apply exit logic
if ticker in open_positions:
position = open_positions[ticker]
entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
exit_price, exit_reason = None, None

# Compute Take Profit (TP) & Stop Loss (SL) prices
tp_price = entry_price * (1 + tp_pct) if direction = = "long" else entry_price * (1 - tp_pct)
sl_price = entry_price * (1 - sl_pct) if direction = = "long" else entry_price * (1 + sl_pct)

# Determine exit conditions
if (direction = = "long" and high > = tp_price) or (direction = = "short" and low < = tp_price):
exit_price, exit_reason = tp_price, "Take Profit"
elif (direction = = "long" and low < = sl_price) or (direction = = "short" and high > = sl_price):
exit_price, exit_reason = sl_price, "Stop Loss"
elif timestamp.astype('datetime64[m]').astype(str)[-5:] > = end_time:
exit_price, exit_reason = close, "EOD Close"

if exit_price is not None:
# Apply slippage
exit_price * = (1 - slippage) if direction = = "long" else (1 + slippage)

# Calculate PnL including brokerage
raw_pnl = (exit_price - entry_price) * qty if direction = = "long" else (entry_price - exit_price) * qty
pnl = raw_pnl - (brokerage * initial_capital * 2)
cumulative_pnl + = pnl

# Store trade details
trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
del open_positions[ticker]
continue

# Open new position
if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
if signal = = 1:
direction = "long"
elif signal = = 2:
direction = "short"
else:
continue

entry_price = open_price * (1 + slippage) if direction = = "long" else open_price * (1 - slippage)
qty = initial_capital / entry_price
open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

return trades

# Main backtesting function
def backtest_trades(data):
"""Runs backtest for all tickers in the dataset using parallel processing."""

# Trading parameters
initial_capital = 100000 # ₹1 lakh per trade
brokerage = 0.001 # 0.1% brokerage
slippage = 0.0001 # 0.01% slippage
tp_pct = 0.02 # 1.5% take-profit
sl_pct = 0.005 # 0.5% stop-loss
end_time = "15:25"
stop_trading_time = "15:15"

# Get unique tickers
unique_tickers = data['Ticker'].unique()

# Convert DataFrame to NumPy structured array for performance
dtype = [
("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"), 
("open", "f8"), ("high", "f8"), ("low", "f8"), 
("PVS", "f8"), ("VMS", "f8"), ("SAMX", "f8"), ("signal", "i4")
]

data_np = np.array(
list(data[["Ticker", "Date", "close", "open", "high", "low", "PVS", "VMS", "SAMX", "signal"]]
.itertuples(index = False, name = None)), 
dtype = dtype
)

# Run backtest in parallel
results = Parallel(n_jobs = -1)(delayed(process_ticker)(ticker, data_np[data_np['Ticker'] = = ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time) for ticker in tqdm(unique_tickers))

# Flatten results into DataFrame
trades = [trade for result in results for trade in result]
return pd.DataFrame(trades, columns = ["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Example usage
# data = pd.read_csv("your_data.csv") # Load dataset
data['Date'] = pd.to_datetime(data['Date']) # Ensure Date column is in datetime format
tradebook = backtest_trades(data)
tradebook.to_csv("/content/drive/MyDrive/tradebook_adanient.csv", index = False)
tradebook